# 03 — Transfer Learning Baseline (ResNet18)

Fine-tunes an ImageNet-pretrained ResNet18 on the same identity subset and split as
the custom CNN, for a fair, direct comparison. Trains in `finetune` mode (all layers
unfrozen) by default — see the second cell to switch to `feature_extract` mode
(frozen backbone, only the new head trained) if you want a third comparison point for
the top rubric tier.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
from torch.utils.data import DataLoader

from dataset import CelebASubset
from models import build_resnet18, num_trainable_params, num_total_params
from utils import set_seed, get_device, train_model

set_seed(42)
device = get_device()
print("device:", device)


In [ ]:
MODE = "finetune"  # or "feature_extract" for a frozen-backbone variant
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4 if MODE == "finetune" else 1e-3  # smaller LR when fine-tuning a pretrained backbone

train_ds = CelebASubset("train")
val_ds = CelebASubset("val")
num_classes = train_ds.num_classes

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
model = build_resnet18(num_classes=num_classes, mode=MODE)
print(f"ResNet18 ({MODE}) total params: {num_total_params(model):,}  trainable: {num_trainable_params(model):,}")


In [ ]:
run_name = f"resnet18_{MODE}"
history, train_time_sec, best_val_acc = train_model(
    model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, device=device, run_name=run_name,
)
print(f"\nBest val accuracy: {best_val_acc:.4f}  |  Training time: {train_time_sec:.1f}s")


**Tip for the highest rubric tier:** re-run this notebook with
`MODE = "feature_extract"` (saved under a different `run_name` automatically) to get
a third architecture variant — a frozen-backbone baseline — to compare against the
fully fine-tuned model and the custom CNN in `04_model_evaluation_comparison.ipynb`.